In [1]:
import pandas as pd

1. Prune the gene list by removing all genes that do not have expression above 1 TPM in at least 3 samples.

In [10]:
tpm_filtered = pd.read_csv("GSE78220_norm_counts_TPM_GRCh38.p13_NCBI.tsv", sep="\t")

mask = (tpm_filtered.iloc[:, 1:] > 1).sum(axis=1) >= 3
# 1: bc the first column is gene names, so we start from index 1
# () > 1: check if TPM is greater than 1
# .sum(axis=1): count how many samples have TPM > 1 for each gene
# >= 3: keep genes that have TPM > 1 in at least 3 samples
# in the end, we get a boolean vector (mask) that indicates which genes pass the filter 
# column: e.g. TRUE TRUE FALSE TRUE ...

filtered_genes = tpm_filtered.loc[mask]
# keep the TRUE rows

print(filtered_genes.shape)
filtered_genes.head()

(20193, 29)


,GeneID,GSM2069823,GSM2069824,GSM2069825,GSM2069826,GSM2069827,GSM2069828,GSM2069829,GSM2069830,GSM2069831,...,GSM2069841,GSM2069842,GSM2069843,GSM2069844,GSM2069845,GSM2069846,GSM2069847,GSM2069848,GSM2069849,GSM2069850
1,653635,14.580,8.5530,16.6500,30.0400,13.6400,23.90,20.870,21.74,23.1100,...,20.0300,13.590,15.620,17.290,18.20,16.1000,27.240,16.99,15.080,24.970
2,102466751,9.791,3.9530,14.9500,6.8840,34.9600,23.60,22.380,16.45,17.9400,...,6.8280,8.803,22.600,12.510,33.23,9.0920,34.460,11.60,7.157,23.430
8,729737,1.238,0.8699,0.5833,0.9341,0.8281,1.98,1.709,3.09,0.9256,...,0.8779,2.277,3.012,7.225,2.13,0.6511,2.671,2.76,0.563,1.530
10,102723897,19.300,7.7200,15.9700,52.8200,17.0600,22.46,26.340,24.30,26.2900,...,21.4400,16.880,20.060,21.410,23.47,18.0900,34.040,24.44,16.470,25.250
11,102465909,5.786,2.8240,9.8690,3.7070,20.7400,10.59,10.780,11.31,8.1920,...,3.0730,3.827,9.453,6.254,13.48,3.7440,13.100,12.18,3.340,9.808


2. Next, obtain the network connecting only the genes that survived the filter in step 1. In this
pruned network, compute the node degree (ND) and the betweenness centrality (BC) for all
nodes that survived Step 1.

In [ ]:
combined_score_threshold = 700
links = pd.read_csv("9606.protein.links.v12.0.txt", sep=" ")
links = links[links["combined_score"] > combined_score_threshold]


print(links.shape)
links.head()

(472000, 3)


,protein1,protein2,combined_score
85,9606.ENSP00000000233,9606.ENSP00000158762,825
130,9606.ENSP00000000233,9606.ENSP00000357048,718
160,9606.ENSP00000000233,9606.ENSP00000262305,952
197,9606.ENSP00000000233,9606.ENSP00000329419,752
268,9606.ENSP00000000233,9606.ENSP00000469035,795


In [32]:
entrez_ids = filtered_genes["GeneID"].tolist()
entrez_ids[:5]

['653635', '102466751', '729737', '102723897', '102465909']

In [ ]:
import mygene

mg = mygene.MyGeneInfo() # so that find the gene symbol (e.g. TP53) for each gene ID (e.g. 7157)

results = mg.querymany(
    entrez_ids,
    scopes="entrezgene", # input IDs are 
    fields="symbol",     # return gene symbols
    species="human"
)

results[:5]


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


In [ ]:
mapping = {}

for r in results:
    if "symbol" in r:
        mapping[r["query"]] = r["symbol"]

list(mapping.items())[:5]

[('653635', 'WASH7P'),
 ('102466751', 'MIR6859-1'),
 ('729737', 'LOC729737'),
 ('102723897', 'LOC102723897'),
 ('102465909', 'MIR6859-2'),
 ('100132287', 'LOC100132287'),
 ('113219467', 'MIR12136'),
 ('100288069', 'LOC100288069'),
 ('79854', 'LINC00115'),
 ('643837', 'LINC01128')]

In [ ]:
filtered_genes["symbol"] = filtered_genes["GeneID"].map(mapping)
filtered_genes[["GeneID", "symbol"]].head()

,GeneID,symbol
1,653635,WASH7P
2,102466751,MIR6859-1
8,729737,LOC729737
10,102723897,LOC102723897
11,102465909,MIR6859-2


In [13]:
import networkx as nx
import mygene

# --- 1. Map Gene Symbols to STRING IDs using mygene ---
mg = mygene.MyGeneInfo()
gene_symbols = filtered_genes.iloc[:, 0].tolist()

# Query mygene for STRING (ensembl protein) IDs
# We filter by 'symbol' and pull 'ensembl.protein'
mapping = mg.querymany(gene_symbols, scopes='symbol', fields='ensembl.protein', species='human')

# Create a clean mapping DataFrame
mapping_df = pd.DataFrame(mapping)

# STRING IDs in your 'links' file usually look like '9606.ENSP...'
# We need to extract the ENSP ID and format it correctly
def format_string_id(row):
    if 'ensembl' in row and isinstance(row['ensembl'], dict):
        protein = row['ensembl'].get('protein')
        if isinstance(protein, list): protein = protein[0]
        return f"9606.{protein}" if protein else None
    return None

mapping_df['string_id'] = mapping_df.apply(format_string_id, axis=1)
mapping_df = mapping_df.dropna(subset=['string_id'])

# Create a dictionary for quick lookup
symbol_to_string = dict(zip(mapping_df['query'], mapping_df['string_id']))
surviving_ids = set(mapping_df['string_id'])


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
19838 input query terms found no hit:	['653635', '102466751', '729737', '102723897', '102465909', '100132287', '113219467', '100288069', '


In [ ]:
# --- 2. Load and Prune the Network with Pandas ---
# Loading only necessary columns to save memory
links_df = pd.read_csv("9606.protein.links.v12.0.txt.gz", sep=" ")

# Prune: Keep only edges where both proteins are in our filtered set
pruned_network = links_df[
    links_df['protein1'].isin(surviving_ids) & 
    links_df['protein2'].isin(surviving_ids)
].copy()

# --- 3. Compute ND and BC using NetworkX ---
G = nx.from_pandas_edgelist(pruned_network, source='protein1', target='protein2')
G.add_nodes_from(surviving_ids) # Include isolated genes

# Calculate metrics
print("Calculating Node Degree...")
node_degree = dict(G.degree())

print("Calculating Betweenness Centrality (this may take a moment)...")
# Note: for very large networks, you can use k=100 in the function to approximate
betweenness_cent = nx.betweenness_centrality(G)

# --- 4. Merge Results into a Final DataFrame ---
# Map everything back to the original Gene Symbols for Step 3/4
results_list = []
for symbol, s_id in symbol_to_string.items():
    results_list.append({
        'Gene_Symbol': symbol,
        'STRING_ID': s_id,
        'ND': node_degree.get(s_id, 0),
        'BC': betweenness_cent.get(s_id, 0.0)
    })

step2_final = pd.DataFrame(results_list)
step2_final.to_csv("step2_metrics.csv", index=False)